In [1]:
# Librerías para la base de Fock, los coeficientes de Bessel y las gráficas.
import numpy as np
import matplotlib.pyplot as plt
from scipy.special import j0
from functools import lru_cache
from scipy.sparse import coo_array
from scipy.sparse.linalg import eigsh


In [2]:
# CASO INICIAL: 6 bosones en 6 sitios. Se mantiene para el espectro y
# los estados fundamentales calculados antes de la última gráfica.
# En construir_espacio_de_Fock(L, N), L cuenta bosones y N cuenta sitios.
def construir_espacio_de_Fock(L, N):

    if N == 1:
        return [(L,)]

    espacio = []

    for n0 in range(L + 1):
        estado_restantes = construir_espacio_de_Fock(L - n0, N - 1)
        for estado in estado_restantes:
            espacio.append((n0,) + estado)

    return espacio



L = 6
N = 6
espacio_de_Fock = np.array(construir_espacio_de_Fock(L, N), dtype=int)

print("Espacio de Fock:")
print(espacio_de_Fock[:10])  # Mostrar solo los primeros 10 estados para no saturar la salida




Espacio de Fock:
[[0 0 0 0 0 6]
 [0 0 0 0 1 5]
 [0 0 0 0 2 4]
 [0 0 0 0 3 3]
 [0 0 0 0 4 2]
 [0 0 0 0 5 1]
 [0 0 0 0 6 0]
 [0 0 0 1 0 5]
 [0 0 0 1 1 4]
 [0 0 0 1 2 3]]


In [3]:
# Energía de interacción local por estado de ocupación, en unidades U = 1.
def E_U(estado):
    return (1/2)*np.sum(estado * (estado - 1))

In [4]:
# Traduce cada tupla de ocupaciones al índice de su vector en la base.
indice_estado = {tuple(estado): i for i, estado in enumerate(espacio_de_Fock)}

In [5]:
# Momentos discretos del caso inicial de seis sitios, con periodicidad.
# Q depende de estos momentos; su caché se vacía al cambiar de sistema.
numero_sitios = N  # N = 6 en el caso inicial.
numeros_de_onda = 2 * np.pi * np.arange(numero_sitios) / numero_sitios


def M(kappa, k_l, k_m, k_n):
    argumento = 2 * kappa * (
        np.cos(k_l)
        + np.cos(k_m)
        - np.cos(k_l + k_m - k_n)
        - np.cos(k_n)
    )
    return j0(argumento)


@lru_cache(maxsize=None)
def Q(kappa, w, x, y, z):
    Q = 0.0 + 0.0j

    for l in range(numero_sitios):
        k_l = numeros_de_onda[l]

        for m in range(numero_sitios):
            k_m = numeros_de_onda[m]

            for n in range(numero_sitios):
                k_n = numeros_de_onda[n]
                fase = (
                    k_l * (x - y)
                    + k_m * (x - z)
                    - k_n * (x - w)
                )
                Q += M(kappa, k_l, k_m, k_n) * np.exp(1j * fase)
    Q *= 1/(2*numero_sitios**3)

    return Q

def construir_tabla_Q(kappa, N):

    tabla_Q = np.empty(
        (N, N, N, N),
        dtype=complex
    )

    for w in range(N):
        for x in range(N):
            for y in range(N):
                for z in range(N):

                    tabla_Q[w, x, y, z] = Q(
                        kappa, w, x, y, z
                    )

    return tabla_Q


In [6]:
# Construcción de H_eff en la base activa: a_w† a_x† a_y a_z.
# La tabla de coeficientes se calcula una vez por Hamiltoniano.
# Los operadores actúan de derecha a izquierda y actualizan las ocupaciones.
@lru_cache(maxsize=None)
def construir_H_eff(kappa):

    dimension = len(espacio_de_Fock)
    filas = []
    columnas = []
    valores = []

    tabla_Q = construir_tabla_Q(kappa, numero_sitios)

    for indice_inicial, estado in enumerate(espacio_de_Fock):

        # Identificamos los sitios que contienen al menos un bosón
        sitios_ocupados = np.flatnonzero(estado)

        # Operadores de destrucción: a_z y a_y
        for z in sitios_ocupados:
            for y in sitios_ocupados:

                # Si y == z, necesitamos al menos dos bosones
                if y == z and estado[z] < 2:
                    continue

                # Operadores de creación: a_x^dagger y a_w^dagger
                for x in range(numero_sitios):
                    for w in range(numero_sitios):

                        resultado = aplicar_operadores(
                            estado, w, x, y, z
                        )

                        if resultado is None:
                            continue

                        nuevo_estado, A = resultado

                        indice_nuevo = indice_estado[
                            tuple(nuevo_estado)
                        ]

                        filas.append(indice_nuevo)
                        columnas.append(indice_inicial)
                        valores.append(tabla_Q[w, x, y, z] * A)

    # Construir la matriz dispersa: CSR usa @ para productos matriciales.
    # Los productos coeficiente * factor bosónico anteriores son escalares.
    # Construir la matriz dispersa
    H_eff = coo_array((valores, (filas, columnas)), shape=(dimension, dimension), dtype=np.complex128).tocsr()
    H_eff.sum_duplicates()  # Combinar entradas duplicadas
    H_eff.eliminate_zeros()  # Eliminar entradas nulas para optimizar el almacenamiento
    return H_eff




def aplicar_operadores(estado, w, x, y, z):

    # Primera destrucción: a_z
    if estado[z] == 0:
        return None

    nuevo_estado = estado.copy()
    A = np.sqrt(nuevo_estado[z])
    nuevo_estado[z] -= 1

    # Segunda destrucción: a_y
    if nuevo_estado[y] == 0:
        return None

    A *= np.sqrt(nuevo_estado[y])
    nuevo_estado[y] -= 1

    # Primera creación: a_x^dagger
    A *= np.sqrt(nuevo_estado[x] + 1)
    nuevo_estado[x] += 1

    # Segunda creación: a_w^dagger
    A *= np.sqrt(nuevo_estado[w] + 1)
    nuevo_estado[w] += 1

    return nuevo_estado, A

In [7]:
# Interacción local de referencia y diagonalización del sistema inicial 6+6.
# H_U no se suma de nuevo a H_eff: se conserva como matriz de referencia.
energias_U = np.array([E_U(estado) for estado in espacio_de_Fock])
H_U = np.diag(energias_U)


    # Aquí no hay productos matriciales: H - H† es una resta y rtol es escalar.


def evaluar_energias(kappa):

    H_eff = construir_H_eff(float(kappa))

    diferencia = H_eff - H_eff.conj().T

    error = (
        np.max(np.abs(diferencia.data))
        if diferencia.nnz else 0.0
    )

    if error > 1e-10:
        raise ValueError(
            f"Hamiltoniano no hermítico: {error:.3e}"
        )

    return np.linalg.eigvalsh(H_eff.toarray()) #muchísmo cuidado que para 8 bosones esto dará problemas, es solo par ala figura 1


In [8]:
# PRIMERA GRÁFICA: espectro completo para 6 bosones y 6 sitios.
numero_puntos_kappa = 21
valores_kappa = np.linspace(0, 1, numero_puntos_kappa)
espectro = np.array(
    [evaluar_energias(float(kappa)) for kappa in valores_kappa]
)

fig, ax = plt.subplots(figsize=(9, 6))

for nivel in espectro.T:
    ax.plot(valores_kappa, nivel, color="tab:blue", linewidth=0.6, alpha=0.45)

ax.set_xlabel(r"$\kappa$")
ax.set_ylabel("Energía")
ax.set_title(r"Espectro de $H_{\mathrm{eff}}(\kappa)$")
ax.set_xlim(0, 1)
ax.grid(alpha=0.25)
fig.tight_layout()
plt.show()

KeyboardInterrupt: 

In [ ]:
# El barrido de esta celda sigue siendo del sistema inicial 6+6.
# La función usa la base activa y se reutiliza después para el caso 8+8.
def obtener_estado_fundamental(kappa):
    H_eff = construir_H_eff(float(kappa))

    diferencia = H_eff - H_eff.conj().T

    error = (
        np.max(np.abs(diferencia.data))
        if diferencia.nnz else 0.0
    )

    if error > 1e-10:
        raise ValueError(
            f"Hamiltoniano no hermítico: {error:.3e}"
        )

    energias, autovectores = eigsh(
        H_eff,
        k=1,
        which="SA",
        tol=1e-9
    )

    E_0 = energias[0]
    estado_fundamental = autovectores[:, 0]

    return E_0, estado_fundamental


resultados_fundamentales = [
    obtener_estado_fundamental(kappa) for kappa in valores_kappa
]

energias_fundamentales = np.array(
    [resultado[0] for resultado in resultados_fundamentales]
)
estados_fundamentales = np.array(
    [resultado[1] for resultado in resultados_fundamentales]
)

In [ ]:
# Correlación de un cuerpo: P(j, l) = <psi_0|a_j† a_l|psi_0>.
# Primero se destruye en l y después se crea en j, también cuando j == l.
def P_optimizada(j, l, estado_fundamental):
    """Calcula <psi_0|a_j† a_l|psi_0> recorriendo la base una sola vez."""
    c = np.asarray(estado_fundamental)
    if c.shape != (len(espacio_de_Fock),):
        raise ValueError("Debe haber una amplitud por estado de Fock.")
    sitios = espacio_de_Fock.shape[1]
    if not (0 <= j < sitios and 0 <= l < sitios):
        raise ValueError(f"Los índices j y l deben estar entre 0 y {sitios - 1}.")

    suma = 0.0 + 0.0j
    for s, estado_s in enumerate(espacio_de_Fock):
        # 1. Sin bosones en l, a_l anula este estado.
        if estado_s[l] == 0:
            continue

        # 2. Copiar las ocupaciones para no modificar la base de Fock.
        nuevo_estado = estado_s.copy()

        # 3. Aplicar a_l: factor sqrt(n_l) y reducir la ocupación en l.
        factor = np.sqrt(nuevo_estado[l])
        nuevo_estado[l] -= 1

        # 4. Aplicar a_j† sobre el estado ya modificado.
        # Esto también trata correctamente j == l, con factor total n_l.
        factor *= np.sqrt(nuevo_estado[j] + 1)
        nuevo_estado[j] += 1

        # 5. Localizar directamente el estado final, sin recorrer toda la base.
        t = indice_estado[tuple(nuevo_estado)]

        # 6. Acumular la contribución escalar a la correlación.
        suma += c[t].conj() * c[s] * factor

    return suma


In [ ]:
# Funciones comunes a ambos sistemas; las dimensiones salen de la base activa.
# Reejecuta esta celda para vaciar la caché si cambias P_optimizada o el Hamiltoniano.
correlaciones_por_kappa = {}


def calcular_correlaciones(kappa, progreso=None):
    kappa = float(kappa)
    sitios = espacio_de_Fock.shape[1]
    if kappa in correlaciones_por_kappa:
        if progreso is not None:
            progreso(sitios * sitios, "Correlaciones recuperadas de memoria")
        return correlaciones_por_kappa[kappa]

    if progreso is not None:
        progreso(0, "Calculando estado fundamental")
    _, psi_0 = obtener_estado_fundamental(kappa)
    matriz = np.zeros((sitios, sitios), dtype=complex)

    for j in range(sitios):
        for l in range(sitios):
            if progreso is not None:
                progreso(j * sitios + l, f"Calculando P({j}, {l})")
            matriz[j, l] = P_optimizada(j, l, psi_0)
            if progreso is not None:
                progreso(j * sitios + l + 1, f"Calculado P({j}, {l})")

    # Solo se guardan matrices completas.
    correlaciones_por_kappa[kappa] = matriz
    return matriz


def rho_desde_correlaciones(k, matriz):
    # Sumatorio sin normalización por el número de sitios; k es el número de onda.
    sitios = matriz.shape[0]
    posiciones = np.arange(sitios)
    diferencias = posiciones[:, None] - posiciones[None, :]
    # Producto elemento a elemento con la matriz de correlaciones, seguido de suma.
    # No es un producto matricial; se conserva * aunque H_eff sea CSR.
    suma = np.sum(np.exp(1j * k * diferencias) * matriz)
    return np.real_if_close(suma)


def rho(k, kappa):
    return rho_desde_correlaciones(k, calcular_correlaciones(kappa))

In [ ]:
# CONFIGURACIÓN EXCLUSIVA DE LA ÚLTIMA GRÁFICA: 8 bosones y 8 sitios.
# Esta celda define el contexto; el cambio se activa al ejecutar el barrido.
# Al salir del contexto, incluso tras un error o una interrupción, se recupera
# la configuración anterior para poder repetir las gráficas del caso 6+6.
from contextlib import contextmanager


@contextmanager
def sistema_ultima_grafica():
    global L, N, espacio_de_Fock, indice_estado
    global numero_sitios, numeros_de_onda, correlaciones_por_kappa

    # Guardamos las referencias originales, sin copiar la base ni los resultados.
    configuracion_anterior = (
        L, N, espacio_de_Fock, indice_estado,
        numero_sitios, numeros_de_onda, correlaciones_por_kappa,
    )
    try:
        # No se pueden reutilizar coeficientes o matrices del caso de seis sitios.
        Q.cache_clear()
        construir_H_eff.cache_clear()
        correlaciones_por_kappa = {}

        L = 8  # Número de bosones para la última gráfica.
        N = 8  # Número de sitios para la última gráfica.
        espacio_de_Fock = np.array(construir_espacio_de_Fock(L, N), dtype=int)
        indice_estado = {tuple(estado): i for i, estado in enumerate(espacio_de_Fock)}

        print("Espacio de Fock:")
        print(espacio_de_Fock[:10])  # Mostrar solo los primeros 10 estados.

        # La base y los momentos deben describir el mismo número de sitios.
        numero_sitios = N
        numeros_de_onda = 2 * np.pi * np.arange(numero_sitios) / numero_sitios
        yield
    finally:
        # Eliminamos la caché de 8+8 antes de recuperar las variables anteriores.
        Q.cache_clear()
        construir_H_eff.cache_clear()
        (
            L, N, espacio_de_Fock, indice_estado,
            numero_sitios, numeros_de_onda, correlaciones_por_kappa,
        ) = configuracion_anterior


In [ ]:
# BARRIDO DE LA ÚLTIMA GRÁFICA: se ejecuta dentro del sistema 8+8.
# R, ks y matrices_correlaciones quedan disponibles al terminar; la base
# y los parámetros generales recuperan después su configuración anterior.
import json
import os
from datetime import datetime
from pathlib import Path
from time import perf_counter

with sistema_ultima_grafica():
    ks = numeros_de_onda.copy()  # Se conservan los ocho momentos para dibujar.
    kappas = np.linspace(0, 1, 21)
    numero_sitios_rho = espacio_de_Fock.shape[1]

    # Un archivo de estado de la última ejecución, consultable sin acceder al kernel.
    archivo_progreso = Path(
        r"C:\Users\Propietario\Downloads\apuntes tfg\progreso_rho.json"
    )
    inicio = perf_counter()
    inicio_fecha = datetime.now().astimezone().isoformat()
    total = len(kappas) * numero_sitios_rho * numero_sitios_rho
    completadas = 0
    indice_kappa = 0
    kappa_actual = None
    estado_ejecucion = "en_curso"

    R = np.full((len(kappas), len(ks)), np.nan, dtype=complex)
    matrices_correlaciones = np.full(
        (len(kappas), numero_sitios_rho, numero_sitios_rho), np.nan, dtype=complex
    )


    def registrar_progreso(fase):
        transcurrido = perf_counter() - inicio
        restante = (
            transcurrido * (total - completadas) / completadas
            if completadas > 0 else None
        )
        datos = {
            "estado": estado_ejecucion,
            "pid": os.getpid(),
            "inicio": inicio_fecha,
            "actualizado": datetime.now().astimezone().isoformat(),
            "fase": fase,
            "indice_kappa": indice_kappa,
            "total_kappas": len(kappas),
            "kappa": kappa_actual,
            "correlaciones_completadas": completadas,
            "correlaciones_totales": total,
            "porcentaje": 100 * completadas / total,
            "segundos_transcurridos": transcurrido,
            "segundos_restantes_estimados": restante,
        }
        temporal = archivo_progreso.with_suffix(".tmp")
        temporal.write_text(json.dumps(datos, indent=2), encoding="utf-8")
        temporal.replace(archivo_progreso)

        estimacion = "calculando" if restante is None else f"{restante / 60:.1f} min"
        print(
            f"\rκ {indice_kappa}/{len(kappas)} | "
            f"P: {completadas}/{total} ({100 * completadas / total:.1f}%) | "
            f"transcurrido: {transcurrido / 60:.1f} min | "
            f"restante aprox.: {estimacion} | {fase}".ljust(170),
            end="", flush=True,
        )


    try:
        registrar_progreso("Inicio del barrido")
        for i, kappa in enumerate(kappas):
            indice_kappa = i + 1
            kappa_actual = float(kappa)
            base = i * numero_sitios_rho * numero_sitios_rho

            def actualizar_correlaciones(locales, fase):
                global completadas
                completadas = base + locales
                registrar_progreso(fase)

            # Solo necesitamos un Hamiltoniano a la vez para esta superficie.
            # Evita acumular las 21 matrices dispersas del sistema 8+8 en memoria.
            construir_H_eff.cache_clear()
            matriz = calcular_correlaciones(kappa, actualizar_correlaciones)
            matrices_correlaciones[i] = matriz
            # Los ocho momentos utilizan la misma matriz: no se vuelve a llamar a P.
            for ik, k in enumerate(ks):
                R[i, ik] = rho_desde_correlaciones(k, matriz)

        R = np.real_if_close(R)
        estado_ejecucion = "completado"
        registrar_progreso("Barrido terminado")
    except KeyboardInterrupt:
        estado_ejecucion = "interrumpido"
        registrar_progreso("Interrumpido por el usuario")
        raise
    except Exception as error:
        estado_ejecucion = "error"
        registrar_progreso(f"{type(error).__name__}: {error}")
        raise
    finally:
        print()

In [ ]:
# ÚLTIMA GRÁFICA: R y ks proceden exclusivamente del barrido 8+8.
# No se usan aquí los momentos generales, que ya vuelven a ser los de 6 sitios.
K, KAPPA = np.meshgrid(ks / np.pi, kappas)

fig = plt.figure()
ax = fig.add_subplot(111, projection="3d")

superficie = ax.plot_surface(
    K, KAPPA, R,
    cmap="viridis",
    edgecolor="none"
)

ax.set_xlabel(r"$k/\pi$")
ax.set_ylabel(r"$\kappa$")
ax.set_zlabel(r"$\rho$")
ax.set_title("8 bosones en 8 sitios")
fig.colorbar(superficie, ax=ax, shrink=0.6)
plt.show()